# Sript de criação de Dataset, treino e Estimação dos Ne Embeddings

## Bibliotecas


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
import logging
import ee
import folium

import numpy as np
from PIL import Image
from matplotlib import pyplot as plt

GPU_AFFINTY  = 0
GPU_MEMORY_LIMIT_GB =8 # For UNET use 8GB

logging.getLogger('googleapicliet.discovery_cache').setLevel(logging.ERROR)
# GPU_AFFINTY  = 0 #GeForce RTX 4090

gpu_dict = {'4090':{'GPU_AFFINTY' : 0, 'GPU_MEMORY_LIMIT_GB':0},
            '2070':{'GPU_AFFINTY':1, 'GPU_MEMORY_LIMIT_GB':8}}


sel_gpu = '4090'
GPU_AFFINTY  = gpu_dict[sel_gpu]['GPU_AFFINTY'] #GeForce RTX 2070
GPU_MEMORY_LIMIT_GB =gpu_dict[sel_gpu]['GPU_MEMORY_LIMIT_GB']

try:
    # ee.Authenticate()
    ee.Initialize(project="ee-claraddays")
except:
    ee.Authenticate()
    ee.Initialize()

EEException: ee.Initialize: no project found. Call with project= or see http://goo.gle/ee-auth.

In [ ]:
EE_TILES = 'https://earthengine.googleapis.com/map/{mapid}/{{z}}/{{x}}/{{y}}?token={token}'

print('Tensorflow Version:',tf.__version__)
print('Folium Version:',folium.__version__)

gpus = tf.config.list_physical_devices('GPU')
print(gpus)
if gpus:
  try:
    tf.config.set_visible_devices(gpus[GPU_AFFINTY], 'GPU')
    GPU_MEMORY_LIMIT_GB = GPU_MEMORY_LIMIT_GB * 1e3
    # Currently, memory growth needs to be the same across GPUs
    if GPU_MEMORY_LIMIT_GB == 0:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        tf.config.set_logical_device_configuration(gpus[GPU_AFFINTY],[tf.config.LogicalDeviceConfiguration(memory_limit=GPU_MEMORY_LIMIT_GB)])
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    # Memory growth must be set before GPUs have been initialized
    print(e)



Tensorflow Version: 2.19.0
Folium Version: 0.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
1 Physical GPUs, 1 Logical GPUs


In [ ]:
print('Tensorflow Version:',tf.__version__)
print(tf.config.list_physical_devices('GPU'))
print(tf.config.list_physical_devices('CPU'))

Tensorflow Version: 2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
def creatDirectory(new_folder):
    if not os.path.exists(new_folder):
        print(f'lets make the directory: {new_folder}')
        os.makedirs(new_folder)
    else: return

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ENV Configs

In [ ]:
# General

VERSION        = '7'
MOSAIC_VERSION = '1'
BIOMA = 'caatinga'
# BIOMA_CODE = 4 # Mata = 4
BUCKET = 'mbX-lulc'
GDRIVE = 'embeddings'
FOLDER = 'training_samples'
TRAINING_BASE = 'training_patches_'+MOSAIC_VERSION
EVAL_BASE     = 'eval_patches_'+MOSAIC_VERSION
ESCALAR_MOSAIC = True

#Local paths
# LOCAL_PATH  = '/mnt/nfs/assets/Mapbiomas/modelos/mbX-resnet-featureMaps'
LOCAL_PATH = '/content/drive/MyDrive/embeddings'
MODEL_DIR   = LOCAL_PATH+'/checkpoint/v'+VERSION
#NFS_PATH = '/mnt/nfs/assets/Mapbiomas/modelos/mbX-resnet-featureMaps'
NFS_PATH = '/content/drive/MyDrive/embeddings'
OUTPUT_PATH = NFS_PATH+'/output/v'+VERSION
ASSET_BIOMAS = 'projects/mapbiomas-workspace/AUXILIAR/biomas_IBGE_250mil'
ASSET_GRID_AMERICAS = 'projects/mapbiomas-raisg/PRODUCTOS/AGUA/DATOS_AUXILIARES/TEMP/sub_grid_0_25_america_sul'
VISUALIZAR_MAP = False
analista_sample = "JULIANO"  # WALLACE, KENIA, LUIZ

# se o folder não esta criado ele cria
creatDirectory(NFS_PATH)
creatDirectory(MODEL_DIR)
creatDirectory(OUTPUT_PATH)

# Exportation Configs
BUCKET_patch = BUCKET
FOLDER_patch = 'allPatch'
FOLDER_classification = 'mbX_lulc_'+VERSION

# Specify inputs (Landsat bands) to the model and the response variable.
opticalBands   =  ['green_median','red_median','nir_median','swir1_median','swir2_median','ndvi_median',\
                   'ndwi_median_wet','slope', 'nir_stdDev'  ]# N-variavel


BANDS    = opticalBands
RESPONSE = 'supervised'
FEATURES = BANDS + [RESPONSE]

# Specify the size and shape of patches expected by the model.
KERNEL_SIZE  = 256
KERNEL_SHAPE = [KERNEL_SIZE, KERNEL_SIZE]
COLUMNS = [
  tf.io.FixedLenFeature(shape=KERNEL_SHAPE, dtype=tf.float32) for k in FEATURES
]
FEATURES_DICT = dict(zip(FEATURES, COLUMNS))

# Sizes of the training and evaluation datasets.
TRAIN_SIZE = 0
EVAL_SIZE = 0

print(BANDS)
# Specify model training parameters.
# BATCH_SIZE  = 64
# DROPOUT     = 0.3 #0.5
# EPOCHS      = 50
# BUFFER_SIZE = 1000
# OPTIMIZER   = 'Nadam'
LOSS        = 'BinaryCrossentropy'
METRICS     = ['BinaryIoU']

['green_median', 'red_median', 'nir_median', 'swir1_median', 'swir2_median', 'ndvi_median', 'ndwi_median_wet', 'slope', 'nir_stdDev']


# Data Visualization

In [ ]:
# modificando o a escala do dado para 8  byte
# https://code.earthengine.google.com/83f2d40ff7484a08ed79f1db86fd9bfb
dict_band = {
    "blue_median": {"min":0, "max": 3731},
    "blue_median_dry": {"min":0, "max": 11492},
    "blue_median_wet": {"min":0, "max": 5268},
    "blue_min": {"min":0, "max": 5268},
    "blue_stdDev": {"min":0, "max": 5097},
    "cai_median": {"min":10000, "max": 61180},
    "cai_median_dry": {"min":10000, "max": 55901},
    "cai_stdDev": {"min":0, "max": 303778},
    "cloud_amp": {"min":0, "max": 242},
    "cloud_max": {"min":0, "max": 242},
    "cloud_median": {"min":0, "max": 65},
    "cloud_median_dry": {"min":0, "max": 207},
    "cloud_median_wet": {"min":0, "max": 93},
    "cloud_min": {"min":0, "max": 93},
    "cloud_stdDev": {"min":0, "max": 93},
    "evi2_amp": {"min":0, "max": 9405},
    "evi2_median": {"min":7250, "max": 18891},
    "evi2_median_dry": {"min":6758, "max": 17430},
    "evi2_median_wet": {"min":7624, "max": 19394},
    "evi2_stdDev": {"min":0, "max": 3674},
    "gcvi_median": {"min":0, "max": 6207990},
    "gcvi_median_dry": {"min":0, "max": 4985173},
    "gcvi_median_wet": {"min":0, "max": 4985173},
    "gcvi_stdDev": {"min":0, "max": 742237},
    "green_median": {"min":0, "max": 4352},
    "green_median_dry": {"min":0, "max": 11088},
    "green_median_texture": {"min":0, "max": 473},
    "green_median_wet": {"min":0, "max": 5011},
    "green_min": {"min":0, "max": 5011},
    "green_stdDev": {"min":0, "max": 4799},
    "gv_amp": {"min":0, "max": 129},
    "gv_max": {"min":0, "max": 130},
    "gv_median": {"min":0, "max": 89},
    "gv_median_dry": {"min":0, "max": 79},
    "gv_median_wet": {"min":0, "max": 109},
    "gv_min": {"min":0, "max": 72},
    "gv_stdDev": {"min":0, "max": 45},
    "gvs_amp": {"min":0, "max": 100},
    "gvs_max": {"min":0, "max": 100},
    "gvs_median": {"min":0, "max": 100},
    "gvs_median_dry": {"min":0, "max": 100},
    "gvs_median_wet": {"min":0, "max": 100},
    "gvs_min": {"min":0, "max": 100},
    "gvs_stdDev": {"min":0, "max": 50},
    "hallcover_median": {"min":1774514, "max": 1849195},
    "hallcover_stdDev": {"min":0, "max": 41449},
    "ndfi_amp": {"min":0, "max": 200},
    "ndfi_max": {"min":0, "max": 200},
    "ndfi_median": {"min":0, "max": 200},
    "ndfi_median_dry": {"min":0, "max": 200},
    "ndfi_median_wet": {"min":0, "max": 200},
    "ndfi_min": {"min":0, "max": 200},
    "ndfi_stdDev": {"min":0, "max": 100},
    "ndvi_amp": {"min":0, "max": 20000},
    "ndvi_median": {"min":0, "max": 20000},
    "ndvi_median_dry": {"min":0, "max": 20000},
    "ndvi_median_wet": {"min":0, "max": 20000},
    "ndvi_stdDev": {"min":0, "max": 9209},
    "ndwi_amp": {"min":0, "max": 20000},
    "ndwi_median": {"min":0, "max": 19999},
    "ndwi_median_dry": {"min":0, "max": 20000},
    "ndwi_median_wet": {"min":0, "max": 19999},
    "ndwi_stdDev": {"min":0, "max": 9058},
    "nir_median": {"min":0, "max": 6406},
    "nir_median_dry": {"min":0, "max": 10213},
    "nir_median_wet": {"min":0, "max": 7196},
    "nir_min": {"min":0, "max": 5296},
    "nir_stdDev": {"min":0, "max": 3382},
    "npv_amp": {"min":0, "max": 29},
    "npv_max": {"min":0, "max": 37},
    "npv_median": {"min":0, "max": 34},
    "npv_median_dry": {"min":0, "max": 32},
    "npv_median_wet": {"min":0, "max": 31},
    "npv_min": {"min":0, "max": 27},
    "npv_stdDev": {"min":0, "max": 11},
    "pri_median": {"min":0, "max": 18434},
    "pri_median_dry": {"min":0, "max": 20000},
    "pri_median_wet": {"min":0, "max": 16930},
    "red_median": {"min":0, "max": 3826},
    "red_median_dry": {"min":0, "max": 10901},
    "red_median_wet": {"min":0, "max": 4781},
    "red_min": {"min":0, "max": 4781},
    "red_stdDev": {"min":0, "max": 4779},
    "savi_median": {"min":6589, "max": 17870},
    "savi_median_dry": {"min":5906, "max": 16783},
    "savi_median_wet": {"min":7082, "max": 18094},
    "savi_stdDev": {"min":0, "max": 3387},
    "sefi_median": {"min":0, "max": 200},
    "sefi_median_dry": {"min":0, "max": 200},
    "sefi_stdDev": {"min":0, "max": 100},
    "shade_amp": {"min":0, "max": 91},
    "shade_max": {"min":22, "max": 129},
    "shade_median": {"min":2, "max": 100},
    "shade_median_dry": {"min":19, "max": 100},
    "shade_median_wet": {"min":3, "max": 100},
    "shade_min": {"min":1, "max": 100},
    "shade_stdDev": {"min":0, "max": 37},
    "slope": {"min":0, "max": 6058},
    "soil_amp": {"min":0, "max": 197},
    "soil_max": {"min":0, "max": 200},
    "soil_median": {"min":0, "max": 75},
    "soil_median_dry": {"min":0, "max": 77},
    "soil_median_wet": {"min":0, "max": 70},
    "soil_min": {"min":0, "max": 62},
    "soil_stdDev": {"min":0, "max": 51},
    "swir1_median": {"min":0, "max": 5255},
    "swir1_median_dry": {"min":0, "max": 5604},
    "swir1_median_wet": {"min":0, "max": 5004},
    "swir1_min": {"min":0, "max": 4741},
    "swir1_stdDev": {"min":0, "max": 1950},
    "swir2_median": {"min":8, "max": 4490},
    "swir2_median_dry": {"min":3, "max": 4719},
    "swir2_median_wet": {"min":0, "max": 4125},
    "swir2_min": {"min":0, "max": 3931},
    "swir2_stdDev": {"min":0, "max": 2984},
    "wefi_amp": {"min":0, "max": 198},
    "wefi_median": {"min":0, "max": 176},
    "wefi_median_wet": {"min":0, "max": 196},
    "wefi_stdDev": {"min":0, "max": 76},
}

coord_lst = [
    [-79.78056664767863,-35.08564354514046],
    [-23.442676022678633,-35.08564354514046],
    [-23.442676022678633,9.555383216755963],
    [-79.78056664767863,9.555383216755963],
    [-79.78056664767863,-35.08564354514046]
]
def scale_image(image):

    img_rescalada = ee.Image().byte()
    for cc, bnd in enumerate(opticalBands):
        print(f" #{cc}  >> {bnd}")
        amplitude = dict_band[bnd]['max'] - dict_band[bnd]['min']
        # recalando de 0 - 1
        img_tmp = image.select(bnd).subtract(dict_band[bnd]['min']).divide(amplitude)
        img_tmp = img_tmp.multiply(255).toByte().rename(bnd)

        # print(img_tmp.reduceRegion(
        #     reducer=ee.Reducer.max(),
        #     geometry= ee.Geometry.Polygon(coord_lst),
        #     scale= 1000,
        #     maxPixels= 1e13,
        #     tileScale= 2
        # ).getInfo())

        img_rescalada = img_rescalada.addBands(img_tmp)

    img_rescalada = img_rescalada.select(opticalBands)

    return img_rescalada




In [ ]:
mosaic_year   = 2022

asset_mapbiomas = 'projects/mapbiomas-public/assets/brazil/lulc/collection10/mapbiomas_brazil_collection10_coverage_v2'
#MUDAR O NEW VALUES OU AGRUPAR AS CLASSES DE INTERESSE E DEIXAR SEQUENCIAL
old_values = [0, 3, 4, 5, 6, 49, 11, 12, 29, 50, 13, 15, 19, 39, 20, 40, 62, 41, 46, 47, 35, 48,  9, 21, 23, 24, 75, 30, 25, 33, 31];
new_values = [0, 1, 2, 3, 4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30];

supervisedImg = (ee.Image(asset_mapbiomas)
                      .select(f'classification_{mosaic_year}')
                      .remap(old_values,new_values).unmask(0)
                      .rename(RESPONSE))
supervisedChannel = supervisedImg.toByte().rename(RESPONSE);


image = (ee.ImageCollection('projects/nexgenmap/MapBiomas2/LANDSAT/BRAZIL/mosaics-2')
                     .filter(ee.Filter.eq('biome', 'CAATINGA'))  #MUDAR para o bioma
                       .filter(ee.Filter.eq('year',  mosaic_year))
                          .mosaic()
        )

if ESCALAR_MOSAIC:
    image = scale_image(image.select(opticalBands))
    print("show band names ", image.bandNames().getInfo())
image = image.addBands(supervisedChannel)



 #0  >> green_median
 #1  >> red_median
 #2  >> nir_median
 #3  >> swir1_median
 #4  >> swir2_median
 #5  >> ndvi_median
 #6  >> ndwi_median_wet
 #7  >> slope
 #8  >> nir_stdDev
show band names  ['green_median', 'red_median', 'nir_median', 'swir1_median', 'swir2_median', 'ndvi_median', 'ndwi_median_wet', 'slope', 'nir_stdDev']


In [ ]:
# Visualizar o mosaico
if VISUALIZAR_MAP:
    mapid = image.getMapId({'bands': ['swir1_median', 'nir_median', 'red_median'], 'min': 0, 'max': 255})
    map = folium.Map(location=[-5.9442, -56.5265])
    folium.TileLayer(
        tiles=mapid['tile_fetcher'].url_format,
        attr='Planet',
        overlay=True,
        name='Mosaic composite',
    ).add_to(map)
    mapid = supervisedChannel.select(RESPONSE).getMapId({'min': 0, 'max': 30 })
    folium.TileLayer(
        tiles=mapid['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='supervisedLayer',
    ).add_to(map)
    map.add_child(folium.LayerControl())
    map

In [ ]:
featureStack = ee.Image.cat([
  image.select(BANDS).unmask(0),
  image.select(RESPONSE).unmask(0)
]).float()
print(featureStack.bandNames().getInfo())
list = ee.List.repeat(1, KERNEL_SIZE)
lists = ee.List.repeat(list, KERNEL_SIZE)
kernel = ee.Kernel.fixed(KERNEL_SIZE, KERNEL_SIZE, lists)

arrays = featureStack.neighborhoodToArray(kernel)

['green_median', 'red_median', 'nir_median', 'swir1_median', 'swir2_median', 'ndvi_median', 'ndwi_median_wet', 'slope', 'nir_stdDev', 'supervised']


MUDAR OS POLÍGONOS DE TREINO/TESTE A DEPENDER DO TEMA/BIOMA

In [ ]:
#WALLACE VERSION (coleta aleatória)
# https://code.earthengine.google.com/acb5486b0100daf878cc17837e47493f
if analista_sample == "WALLACE":   # WALLACE, KENIA, LUIZ
    def carregar_cartas_com_random(asset_path):
        # parâmetros fixos
        random_col_name = 'random'
        numeric_col_name = 'random_int'
        max_value = 100

        ASSET_BIOMAS = 'projects/mapbiomas-workspace/AUXILIAR/biomas_IBGE_250mil'
        nameBiomas = ["Cerrado", "Mata Atlântica", "Pampa", "Pantanal", "Amazônia", "Caatinga"]
        bioma_name = nameBiomas[5]  # selecionando a Caatinga
        bioma = (ee.FeatureCollection(ASSET_BIOMAS)
                    .filter(ee.Filter.eq('Bioma', bioma_name))
                    .geometry()
                )

        fc = (ee.FeatureCollection(asset_path)
                        .filterBounds(bioma)
                        .map(lambda f: ee.Feature(f.geometry()))
            )
        fc = fc.randomColumn(random_col_name)
        fc = fc.map(lambda f: f.set(numeric_col_name,f.getNumber(random_col_name).multiply(max_value).add(1).int()))
        return fc


    asset_cartas_ref = 'projects/mapbiomas-workspace/AUXILIAR/CARTAS_IBGE/articulacao_100000_mapbiomas'
    cartas = carregar_cartas_com_random(asset_cartas_ref)
    # Divide em 30% e 70%
    evalPolys = cartas.filter(ee.Filter.lt('random', 0.3))
    trainingPolys = cartas.filter(ee.Filter.gte('random', 0.3))

In [ ]:
# KENIA VERSION (coleta manual)
# https://code.earthengine.google.com/f27b95cf04772b06613b783ceacd6146
if analista_sample == "KENIA":   # WALLACE, KENIA, LUIZ
    merged = ee.FeatureCollection('users/kenia_remapgeo/tiles_mobilenetV3_teste0')

    trainingPolys =  merged.filter(ee.Filter.eq('tipo', 'train'))
    evalPolys = merged.filter(ee.Filter.eq('tipo', 'test'))

In [ ]:
# LUIZ VERSION (coleta manual sem código)
if analista_sample == "LUIZ":   # WALLACE, KENIA, LUIZ
    trainingPolys =  ee.FeatureCollection.loadBigQueryTable('solved-mb10.mb10_database.amostras_mbx_treino','geo')
    evalPolys = ee.FeatureCollection.loadBigQueryTable('solved-mb10.mb10_database.amostras_mbx_teste','geo')

In [ ]:
# JULIANO VERSION (SORTEIO ALEATÓRIO EM GRID)
# https://code.earthengine.google.com/59c33d789094cba129bc2fc24b281137
#falta exportar a fc
if analista_sample == "JULIANO":   # WALLACE, KENIA, LUIZ
    def selec_set_polygons__train_test(feat_col):
        #
        feat_col = feat_col.randomColumn('split_regions')
        feat_col = feat_col.randomColumn('split_samples')
        # selecionando a mitade dos grides
        feat_col = feat_col.filter(ee.Filter.lt('split_regions', 0.5))
        # exportando os poligons de treino e Teste
        polg_train = feat_col.filter(ee.Filter.lt("split_samples", 0.7));  # 70% para traino
        polg_test = feat_col.filter(ee.Filter.gte("split_samples", 0.7));  # 30% para teste
        return polg_train, polg_test


    nameBiomas = ["Cerrado", "Mata Atlântica", "Pampa", "Pantanal", "Amazônia", "Caatinga"]
    bioma_name = nameBiomas[5]  # selecionando a Caatinga
    bioma = (ee.FeatureCollection(ASSET_BIOMAS)
                .filter(ee.Filter.eq('Bioma', bioma_name))
                .geometry()
            )
    ## selecionar as grides do bioma
    subgrids = (ee.FeatureCollection(ASSET_GRID_AMERICAS)
                    .filterBounds(bioma))

    trainingPolys, evalPolys = selec_set_polygons__train_test(subgrids)

In [ ]:
# show polygons numbers
print("número de poligonos Training ", trainingPolys.size().getInfo())
print("número de poligonos Teste ", evalPolys.size().getInfo())
print("show matedata \n >> ", evalPolys.first().getInfo())



número de poligonos Training  453
número de poligonos Teste  187
show matedata 
 >>  {'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-35.99999946546712, -5.499997906138102], [-35.99999946546712, -5.249998443740369], [-36.12499920124301, -5.249998440136998], [-36.24999895181556, -5.249998427428455], [-36.249998951815556, -5.499997952286506], [-36.12499920124301, -5.499997911738181], [-35.99999946546712, -5.499997906138102]]]}, 'id': '00000000000000002eea', 'properties': {'bottom': -5.5, 'id': 82678, 'left': -36.25, 'right': -36, 'split_regions': 0.4293857680185026, 'split_samples': 0.8404166855441794, 'top': -5.25}}


In [ ]:
if VISUALIZAR_MAP:
    polyImage = ee.Image(0).byte().paint(trainingPolys, 1).paint(evalPolys, 2)
    polyImage = polyImage.updateMask(polyImage)
    mapid = polyImage.getMapId({'min': 1, 'max': 2, 'palette': ['red', 'blue']})
    map = folium.Map(location=[-1.3621, -45.2738], zoom_start=5)

    # mapid = supervisedChannel.select(RESPONSE).getMapId({'min': 0, 'max': 1, 'pallete':'#ff0000'})
    folium.TileLayer(
        tiles=mapid['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='supervisedLayer',
    ).add_to(map)
    map.add_child(folium.LayerControl())
    folium.TileLayer(
        tiles=mapid['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        overlay=True,
        name='training polygons',
    ).add_to(map)

    map

In [ ]:
# MUDAR O BIOMA E A VERSÃO
TRAINING_BASE= 'training_patches_'+str(VERSION)+'_'+BIOMA
# TRAINING_BASE
# EVAL_BASE
EVAL_BASE = 'eval_patches_'+str(VERSION)+'_'+BIOMA
FOLDER_TRAIN = 'training_samples_v'+str(VERSION)+'_MB10_'+BIOMA
FOLDER_EVAL = 'eval_samples_v'+str(VERSION)+'_MB10_'+BIOMA
print("folder driver ===> ", GDRIVE)

folder driver ===>  embeddings


# Train/Test Chips Exportation

In [ ]:
# Convert the feature collections to lists for iteration.
trainingPolysList = trainingPolys.toList(trainingPolys.size())
evalPolysList = evalPolys.toList(evalPolys.size())
# These numbers determined experimentally.
n = 10 # Number of shards in each polygon.
N = 100 # Total sample size in each polygon.

#Add some generalism
TRAIN_SIZE = trainingPolys.size().getInfo()*N
EVAL_SIZE = evalPolys.size().getInfo()*N
print('TRAIN:'+str(TRAIN_SIZE))
print('EVAL:'+str(EVAL_SIZE))

# Export all the training data (in many pieces), with one task
# per geometry.
for g in range(trainingPolys.size().getInfo()):
  geomSample = ee.FeatureCollection([])
  for i in range(n):
    sample = arrays.sample(
      region = ee.Feature(trainingPolysList.get(g)).geometry(),
      scale = 30,
      numPixels = N / n, # Size of the shard.
      seed = i,
    #  tileScale = 8
    )
    geomSample = geomSample.merge(sample)

  desc = TRAINING_BASE + '_g' + str(g)
  task = ee.batch.Export.table.toDrive(
    collection = geomSample,
    description = desc,
    folder = GDRIVE+'_'+FOLDER_TRAIN,
    fileNamePrefix = desc,
    fileFormat = 'TFRecord',
    selectors = BANDS + [RESPONSE]
  )
  task.start()

# Export all the evaluation data.
for g in range(evalPolys.size().getInfo()):
  geomSample = ee.FeatureCollection([])
  for i in range(n):
    sample = arrays.sample(
      region = ee.Feature(evalPolysList.get(g)).geometry(),
      scale = 30,
      numPixels = N / n,
      seed = i,
    #  tileScale = 8
    )
    geomSample = geomSample.merge(sample)

  desc = EVAL_BASE + '_g' + str(g)
  task = ee.batch.Export.table.toDrive(
    collection = geomSample,
    description = desc,
    folder = GDRIVE+'_'+FOLDER_EVAL,
    fileNamePrefix = desc,
    fileFormat = 'TFRecord',
    selectors = BANDS + [RESPONSE],
  )
  task.start()

TRAIN:45300
EVAL:18700


SCRIPT AJUSTADO SÓ ATÉ ESSE PONTO

In [ ]:
# TRAINING_BASE = 'training_patches_4_MG'
TRAINING_BASE

'training_patches_7_pantanal'

In [ ]:
def parse_tfrecord(example_proto):
  """The parsing function.
  Read a serialized example into the structure defined by FEATURES_DICT.
  Args:
    example_proto: a serialized Example.
  Returns:
    A dictionary of tensors, keyed by feature name.
  """
  print(FEATURES_DICT)
  return tf.io.parse_single_example(example_proto, FEATURES_DICT)


def to_tuple(inputs):
  """Function to convert a dictionary of tensors to a tuple of (inputs, outputs).
  Turn the tensors returned by parse_tfrecord into a stack in HWC shape.
  Args:
    inputs: A dictionary of tensors, keyed by feature name.
  Returns:
    A dtuple of (inputs, outputs).
  """
  inputsList = [inputs.get(key) for key in FEATURES]
  stacked = tf.stack(inputsList, axis=0)
  # Convert from CHW to HWC
  stacked = tf.transpose(stacked, [1, 2, 0])
  return stacked[:,:,:len(BANDS)], stacked[:,:,len(BANDS):]


def get_dataset(pattern):
  """Function to read, parse and format to tuple a set of input tfrecord files.
  Get all the files matching the pattern, parse and convert to tuple.
  Args:
    pattern: A file pattern to match in a Cloud Storage bucket.
  Returns:
    A tf.data.Dataset
  """
  glob = tf.io.gfile.glob(pattern)
  dataset = tf.data.TFRecordDataset(glob, compression_type='GZIP',num_parallel_reads=8)
  dataset = dataset.map(parse_tfrecord, num_parallel_calls=8)
  dataset = dataset.map(to_tuple, num_parallel_calls=8)
  return dataset

In [ ]:
from tensorflow.keras import preprocessing
import random

TRAIN_SIZE = 0

BATCH_SIZE = 32

# It's good practice to have this helper function separate
def squeeze_mask(image, mask):
  """Ensures the mask has the shape [H, W] and not [H, W, 1]."""
  with tf.device('/gpu:0'):
      mask = tf.squeeze(mask, axis=-1)
      mask = tf.cast(mask, tf.uint8)
      return image, mask

def augment_spatial(image, label):
    """Randomly translates/pads the image."""
    # This function expects a 2D label, so we don't need to squeeze inside here anymore.
    with tf.device('/gpu:0'):
    # We add a temporary channel dimension for padding, then remove it.
        label = label[..., tf.newaxis] # Temporarily add back the channel: (256, 256, 1)

        padded_image = tf.pad(image, [[64, 64], [64, 64], [0, 0]], mode='CONSTANT')
        padded_label = tf.pad(label, [[64, 64], [64, 64], [0, 0]], mode='CONSTANT')

        random_x = tf.random.uniform([], 0, 129, dtype=tf.int32)
        random_y = tf.random.uniform([], 0, 129, dtype=tf.int32)

        cropped_image = tf.slice(padded_image, [random_x, random_y, 0], [256, 256, 6])
        cropped_label = tf.slice(padded_label, [random_x, random_y, 0], [256, 256, 1])

        # Squeeze the label again after cropping to return a 2D tensor
        cropped_label = tf.squeeze(cropped_label, axis=-1)

        return cropped_image, cropped_label

def removeNan(image,label):
    image = tf.keras.ops.nan_to_num(image)
    label = tf.keras.ops.nan_to_num(label)
    return image,label

def get_training_dataset():
    TRAINING_BASE = "training_patches_1"
    #MUDAR PARA O CAMINHO DA SUA PASTA NO DRIVE
    #glob = '/mnt/nfs/assets/Mapbiomas/modelos/mbX-resnet-featureMaps/train/*.tfrecord.gz'
    glob = './train_samples_v1/*.tfrecord.gz'
    dataset = get_dataset(glob)
    squeezed_dataset = dataset.map(squeeze_mask, num_parallel_calls=tf.data.AUTOTUNE)
    # TRAIN_SIZE = squeezed_dataset.reduce(np.int64(0), lambda x, _: x + 1).numpy()
    # print(TRAIN_SIZE * 2)
    augmented_dataset = squeezed_dataset.map(augment_spatial, num_parallel_calls=tf.data.AUTOTUNE)
    final_dataset = augmented_dataset.concatenate(squeezed_dataset)
    final_dataset = final_dataset.shuffle(20000, reshuffle_each_iteration=True).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)
    return final_dataset.map(removeNan, num_parallel_calls=tf.data.AUTOTUNE)

training = get_training_dataset()
#first_element = next(iter(training))

{'green_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'red_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir1_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir2_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'ndvi_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'ndwi_median_wet': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'slope': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir_stdDev': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'supervised': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None)}


In [ ]:
import tensorflow as tf
import numpy as np

# You still need this helper function to fix the mask shape
def squeeze_and_cast_mask(image, mask):
    """Ensures the mask has the shape [H, W] and not [H, W, 1]."""
    with tf.device('/gpu:0'):
        mask = tf.squeeze(mask, axis=-1)
        mask = tf.cast(mask, tf.uint8)
        return image, mask


def get_eval_dataset():
    """
    Creates the evaluation dataset pipeline.
    NO augmentation, NO concatenation, NO shuffle, NO repeat.
    """
    glob = './eval_samples_v1/*.tfrecord.gz'

    # 1. Get the initial dataset
    dataset = get_dataset(glob) # Assumes this function exists and parses TFRecords

    # 2. Apply the squeeze and cast operation to ensure correct mask shape
    dataset = dataset.map(squeeze_and_cast_mask, num_parallel_calls=tf.data.AUTOTUNE)

    # 3. Batch the data and prefetch for performance
    # The BATCH_SIZE here can be the same or different from your training batch size
    final_dataset = dataset.batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)

    return final_dataset.map(removeNan, num_parallel_calls=tf.data.AUTOTUNE)

# You would then create your evaluation dataset like this:
evaluation = get_eval_dataset()

{'green_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'red_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir1_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir2_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'ndvi_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'ndwi_median_wet': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'slope': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir_stdDev': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'supervised': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None)}


# Model Instantiation

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV3Small,MobileNetV3Large
from tensorflow.keras.layers import Conv2D, Input, UpSampling2D
from tensorflow.keras.models import Model
from tensorflow.keras.metrics import MeanIoU

# IDEIAS AQUI, USAR QKERAS e converter tudo para quantização de 4BITS
# DEIXAR A ARQUITETURA DINAMICA, RECEBENDO A QUANTIDADE DE BANDAS DE MANEIRA DINAMICA E SEM PULOS BRUTOS NA REDUÇÃO

print("--- Building the full Encoder-Decoder model in a single run ---")

# --- Part 1: Build the Encoder (your custom feature extractor) ---
nclasses = 31
base_model = MobileNetV3Large( include_top=False, input_shape=(256, 256, 3),classes= nclasses) #weights='imagenet',
connection_point_name = 'conv_bn'
connection_layer = base_model.get_layer(connection_point_name)
tail_model = Model(inputs=connection_layer.input, outputs=base_model.output, name='mobilenet_tail')
new_input = Input(shape=(256, 256, len(BANDS)), name='6_band_input')
new_first_layer = Conv2D(16, (3, 3), strides=(2, 2), padding='same', name='new_first_conv')(new_input)
encoder_output = tail_model(new_first_layer)
encoder = Model(inputs=new_input, outputs=encoder_output, name='encoder')

print("Encoder built successfully. Output shape:", encoder.output.shape)

# --- Part 2: Build the Decoder (NEW PART) ---

# Shape is (None, 8, 8, 576) from your model summary
x = encoder.output

# Upsample 8x8 -> 16x16
x = UpSampling2D(size=(2, 2))(x)
x = Conv2D(512, (3, 3), activation='relu', padding='same')(x) #it was 256, v6 in below

# Upsample 16x16 -> 32x32
x = UpSampling2D(size=(2, 2))(x)
x = Conv2D(256, (3, 3), activation='relu', padding='same')(x) #it was 128, v6 in below

# Upsample 32x32 -> 64x64
x = UpSampling2D(size=(2, 2))(x)
x = Conv2D(128, (3, 3), activation='relu', padding='same')(x) #it was 64, v6 in below

# Upsample 64x64 -> 128x128
x = UpSampling2D(size=(2, 2))(x)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(x) #it was 32, v6 in below

# Upsample 128x128 -> 256x256
x = UpSampling2D(size=(2, 2))(x)
x = Conv2D(32, (3, 3), activation='relu', padding='same')(x) #it was 16, v6 in below

print("Decoder path built. Shape before final prediction:", x.shape)


# --- Part 3: Add the final segmentation head ---

num_classes = 76#51
final_output = Conv2D(num_classes, (1, 1), activation='softmax', name='segmentation_head')(x)

print("Final output shape:", final_output.shape)


# --- Part 4: Create and Compile the final model ---

full_segmentation_model = Model(inputs=encoder.input, outputs=final_output)

# full_segmentation_model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss='sparse_categorical_crossentropy',
#     metrics=[MeanIoU(num_classes=num_classes, name='mean_iou')]
# )

print("\nFull Encoder-Decoder model compiled successfully!")
full_segmentation_model.summary()

--- Building the full Encoder-Decoder model in a single run ---


/usr/local/lib/python3.12/dist-packages/keras/src/applications/mobilenet_v3.py:517: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Encoder built successfully. Output shape: (None, 8, 8, 960)
Decoder path built. Shape before final prediction: (None, 256, 256, 32)
Final output shape: (None, 256, 256, 76)

Full Encoder-Decoder model compiled successfully!


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 6_band_input (InputLayer)       │ (None, 256, 256, 9)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_first_conv (Conv2D)         │ (None, 128, 128, 16)   │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_tail (Functional)     │ (None, 8, 8, 960)      │     2,995,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_10 (UpSampling2D) │ (None, 16, 16, 960)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 16, 16, 512)    │     4,424,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_11 (UpSampling2D) │ (None, 32, 32, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 32, 32, 256)    │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_12 (UpSampling2D) │ (None, 64, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 64, 64, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_13 (UpSampling2D) │ (None, 128, 128, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 128, 128, 64)   │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_14 (UpSampling2D) │ (None, 256, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 256, 256, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ segmentation_head (Conv2D)      │ (None, 256, 256, 76)   │         2,508 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,991,132 (34.30 MB)

 Trainable params: 8,966,732 (34.21 MB)

 Non-trainable params: 24,400 (95.31 KB)

# Model Selection/Load

In [ ]:
from IPython.display import Image
from tensorflow.keras import metrics
from tensorflow.keras import optimizers
from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras.metrics import MeanIoU


model = full_segmentation_model # MobileNetv3

EPOCH = 25
# CHECK_MODEL_DIR_MG = '/mnt/nfs/assets/Mapbiomas/modelos/mbX-resnet-featureMaps/checkpoint/v5/cp-00'+str(EPOCH)+'.keras'
#MUDAR PARA PASTA NO DRIVE
CHECK_MODEL_DIR_MG = '/content/drive/MyDrive/embeddings/checkpoint/v7/cp-00'+str(EPOCH)+'.keras'
#model.load_weights(CHECK_MODEL_DIR_MG)
print(model.summary())


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 6_band_input (InputLayer)       │ (None, 256, 256, 9)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_first_conv (Conv2D)         │ (None, 128, 128, 16)   │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_tail (Functional)     │ (None, 8, 8, 960)      │     2,995,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_10 (UpSampling2D) │ (None, 16, 16, 960)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 16, 16, 512)    │     4,424,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_11 (UpSampling2D) │ (None, 32, 32, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 32, 32, 256)    │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_12 (UpSampling2D) │ (None, 64, 64, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 64, 64, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_13 (UpSampling2D) │ (None, 128, 128, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 128, 128, 64)   │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_14 (UpSampling2D) │ (None, 256, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 256, 256, 32)   │        18,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ segmentation_head (Conv2D)      │ (None, 256, 256, 76)   │         2,508 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,991,132 (34.30 MB)

 Trainable params: 8,966,732 (34.21 MB)

 Non-trainable params: 24,400 (95.31 KB)

None


In [ ]:
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt

def previewClass(epoch,log):
    counter = 0
    for batch in evaluation.shuffle(10).take(3):
        pureImage = batch[0]
        supervised = batch[1]
        stacked = tf.transpose(pureImage[0], [0, 1, 2]).numpy()
        stackedS = tf.transpose(supervised[0], [0, 1]).numpy()
        #tf.contrib.summary.image(str(epoch)+" ->Input/"+str(counter), stacked[0:3], step=0)

        test_pred_raw = model.predict(pureImage)
        test_pred_raw = tf.argmax(test_pred_raw, axis=-1)
        test_pred_raw = tf.transpose(test_pred_raw[0],[0, 1]).numpy()
        fig = plt.figure(figsize=[12,4])
        # show original image
        fig.add_subplot(131)
        plt.imshow(stacked[:,:,0:3].astype(np.uint8), interpolation='nearest', vmin=0, vmax=255)
        fig.add_subplot(132)
        plt.imshow(stackedS[:,:], interpolation='nearest',cmap="gray")
        fig.add_subplot(133)
        plt.imshow(test_pred_raw[:,:], interpolation='nearest',cmap="gray")
        plt.show()
        #tf.contrib.summary.image(str(epoch)+" -> Out/"+str(counter), test_pred_raw, step=0)
        counter = counter+1
# previewClass(1,1)

In [ ]:
# previewClass(1,1)

In [ ]:
creatDirectory(MODEL_DIR)

# DATASET INTEGRITY CHECK

In [ ]:
import tensorflow as tf
import numpy as np

# --- Helper function (remains the same) ---
def squeeze_mask(image, mask):
    """Ensures the mask has the shape [H, W] and not [H, W, 1]."""
    mask = tf.squeeze(mask, axis=-1)
    return image, mask

# Assuming get_dataset(glob) is defined elsewhere and works
# from your_module import get_dataset

def run_integrity_check(name, file_glob):
    """
    Iterates through an entire dataset to find shape inconsistencies AND
    invalid numerical values (NaN or Inf).
    """
    print(f"\n--- Starting Full Integrity Check for: {name} ---")

    # 1. Create a dataset pipeline WITHOUT shuffle, batch, or prefetch
    raw_dataset = get_dataset(file_glob)
    # NOTE: We cast to float32 here to check for NaN/Inf before any other processing
    processed_dataset = raw_dataset.map(lambda img, msk: (tf.cast(img, tf.float32), tf.cast(msk, tf.float32)))

    error_found = False
    # 2. Enumerate allows us to see which example number is bad
    for i, (image, mask) in enumerate(processed_dataset):

        # --- NEW: Check for NaN or Inf values ---
        if tf.reduce_any(tf.math.is_nan(image)):
            print(f"!!! NaN VALUE FOUND in IMAGE at example index: {i} !!!")
            error_found = True
            break
        if tf.reduce_any(tf.math.is_inf(image)):
            print(f"!!! Inf VALUE FOUND in IMAGE at example index: {i} !!!")
            error_found = True
            break
        if tf.reduce_any(tf.math.is_nan(mask)):
            print(f"!!! NaN VALUE FOUND in MASK at example index: {i} !!!")
            error_found = True
            break
        if tf.reduce_any(tf.math.is_inf(mask)):
            print(f"!!! Inf VALUE FOUND in MASK at example index: {i} !!!")
            error_found = True
            break
        # --- End of new check ---

        # Squeeze the mask for shape checking
        image, mask = squeeze_mask(image, mask)
        image_shape = image.shape
        mask_shape = mask.shape

        # 3. Check if shapes are correct for every single example
        if len(image_shape) != 3 or image_shape[0] != 256 or image_shape[1] != 256 or image_shape[2] != 6:
            print(f"!!! CORRUPTED IMAGE SHAPE FOUND at example index: {i} !!!")
            print(f"    Expected shape: (256, 256, 6), but got: {image_shape}")
            error_found = True
            break

        if len(mask_shape) != 2 or mask_shape[0] != 256 or mask_shape[1] != 256:
            print(f"!!! CORRUPTED MASK SHAPE FOUND at example index: {i} !!!")
            print(f"    Expected shape: (256, 256), but got: {mask_shape}")
            error_found = True
            break

        # Optional: Print progress every 1000 steps
        if (i + 1) % 1000 == 0:
            print(f"    Checked {i+1} examples... OK")

    if not error_found:
        print(f"--- Full {name} dataset check PASSED. No shape or invalid value errors found. ---")
    else:
        print(f"--- {name} dataset check FAILED. Please investigate the corrupted record. ---")

# --- Run the check on both your datasets ---
training_glob = '/content/drive/MyDrive/embeddings_training_samples_v7_MB10_pantanal/*.tfrecord.gz'
eval_glob = '/content/drive/MyDrive/embeddings_eval_samples_v7_MB10_pantanal/*.tfrecord.gz' #<-- MAKE SURE THIS PATH IS CORRECT

run_integrity_check("Training Data", training_glob)
run_integrity_check("Evaluation Data", eval_glob)


--- Starting Full Integrity Check for: Training Data ---
{'green_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'red_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir1_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir2_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'ndvi_median': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'ndwi_median_wet': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'slope': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir_stdDev': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'supervised': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None)}
!!! CORRUPTED IMAGE SHAPE FOUND at example index: 0 !!!
    Expe

In [ ]:
print("\n--- Verifying Label Range ---")
# Create a dataset WITHOUT batching to check individual masks
raw_training_data = get_dataset('/mnt/nfs/assets/Mapbiomas/modelos/mbX-resnet-featureMaps/train/*.tfrecord.gz') # Use your glob
squeezed_training_data = raw_training_data.map(squeeze_mask)

min_label = 999
max_label = -999

# This loop might take a minute
for i, (image, mask) in enumerate(squeezed_training_data):
    current_min = tf.reduce_min(mask)
    current_max = tf.reduce_max(mask)
    if current_min < min_label:
        min_label = current_min
    if current_max > max_label:
        max_label = current_max
    if (i + 1) % 1000 == 0:
        print(f"Scanned {i+1} masks...")

print(f"\nScan Complete. Minimum label found: {min_label.numpy()}")
print(f"Scan Complete. Maximum label found: {max_label.numpy()}")

In [ ]:
raw_training_data = get_dataset('/mnt/nfs/assets/Mapbiomas/modelos/mbX-resnet-featureMaps/train/*.tfrecord.gz') # Use your glob
squeezed_training_data = raw_training_data.map(squeeze_mask)

for image, mask in squeezed_training_data.take(1):
    print("Mask dtype:", mask.dtype)
    print("Mask shape:", mask.shape)
    print("Mask unique values:", tf.unique(tf.reshape(mask, [-1]))[0][:50])

# TRAIN

In [ ]:
# --- THE ISOLATION EXPERIMENT ---
# After your full_segmentation_model has been built...

print("--- LAUNCHING ISOLATION EXPERIMENT: Compiling without MeanIoU ---")

checkpoint_path = MODEL_DIR+"/cp-{epoch:04d}.keras"
checkpoint_dir = os.path.dirname(checkpoint_path)
tensorboard = tf.keras.callbacks.TensorBoard(log_dir='output/v'+VERSION+'/log_model', write_images=True)

cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    verbose=1,
    save_weights_only=False,
    save_best_only=False
)

# Add clipnorm=1.0 to the optimizer
full_segmentation_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nModel re-compiled without MeanIoU. Attempting to fit...")

# Re-create your datasets
training = get_training_dataset()
evaluation = get_eval_dataset()

# Now, run the fit command exactly as before
# Let's just run for a few epochs to see if it starts
with tf.device('/gpu:0'):
    result = full_segmentation_model.fit(
        training,
        epochs=200,
        initial_epoch=0,
        verbose=1,
        validation_data=evaluation,
        callbacks=[cp_callback, tensorboard]
    )

--- LAUNCHING ISOLATION EXPERIMENT: Compiling without MeanIoU ---

Model re-compiled without MeanIoU. Attempting to fit...
{'green': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'red': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir1': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'NDVI': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'MNDWI': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'supervised': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None)}
{'green': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'red': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'nir': FixedLenFeature(shape=[256, 256], dtype=tf.float32, default_value=None), 'swir1': FixedLenFeature(shape=[256, 256], dtype=tf.float32

I0000 00:00:1753294469.269128 3749520 service.cc:152] XLA service 0x7d03bc0029d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1753294469.269229 3749520 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
I0000 00:00:1753294475.799472 3749520 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1753294515.647871 3749520 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   2289/Unknown 577s 194ms/step - accuracy: 0.7463 - loss: 0.9000

/usr/lib/python3.12/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(value)



Epoch 1: saving model to /mnt/nfs/assets/Mapbiomas/modelos/mbX-resnet-featureMaps/checkpoint/v7/cp-0001.keras
2289/2289 ━━━━━━━━━━━━━━━━━━━━ 675s 237ms/step - accuracy: 0.7464 - loss: 0.8999 - val_accuracy: 0.6891 - val_loss: 1.0421
Epoch 2/200


In [ ]:
previewClass(1,1)

In [ ]:
import tensorflow as tf
import numpy as np

print("--- Verifying Data Normalization ---")
# Re-create your dataset to be sure
training_dataset_for_check = get_training_dataset()

# Take one batch
image_batch, mask_batch = next(iter(training_dataset_for_check))

# Convert to NumPy and check the range
image_batch_np = image_batch.numpy()

print(f"Image batch data type: {image_batch_np.dtype}")
print(f"Min value in image batch: {np.min(image_batch_np)}")
print(f"Max value in image batch: {np.max(image_batch_np)}")
print(f"Mean value in image batch: {np.mean(image_batch_np)}")

# MODEL LOAD AND FEATURE DECODER SELECTION

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model

# --- Step 1: Load your final, fully trained model ---
print("Loading final trained model...")
# trained_model = tf.keras.models.load_model('final_mapbiomas_segmentation_model.keras')


trained_model = model
print("Model loaded.")
# --- Step 2: CHOOSE THE TARGET LAYER NAME ---
# Replace this string with the name of the layer you want from the model summary.
# Example1: Let's choose 'conv2d_2' to get the 64x64x64 features.
# Example2: Let's choose 'conv2d_1' to get the 32x32x128 features.
# Example3: Let's choose 'conv2d_3' to get the 128x128x32 features.
TARGET_LAYER_NAME = 'conv2d_2'


# --- Step 3: Create the new feature extractor model ---
print(f"\nCreating feature extractor ending at layer '{TARGET_LAYER_NAME}'...")
try:
    # Define the start and end points of the "slice"
    input_tensor = trained_model.input
    output_tensor = trained_model.get_layer(TARGET_LAYER_NAME).output

    # Create the new model. It will include all layers from the input to your target.
    new_feature_extractor = Model(
        inputs=input_tensor,
        outputs=output_tensor,
        name=f"extractor_until_{TARGET_LAYER_NAME}"
    )

    print("--- Successfully created new extractor ---")
    new_feature_extractor.summary()

except ValueError:
    print(f"\n--- ERROR ---")
    print(f"Could not find a layer named '{TARGET_LAYER_NAME}'.")
    print("Please check the name in your model summary and try again.")

# Prediction

In [ ]:
# VERSION     = '2_BR_append_4'
OUTPUT_PATH =  LOCAL_PATH+'/output/v'+VERSION
creatDirectory(OUTPUT_PATH)

In [ ]:
print(OUTPUT_PATH)
print(VERSION)
print(EPOCH)

In [ ]:
import rasterio
import numpy as np
import tensorflow as tf
from rasterio.transform import Affine
from patchify import patchify
import os
from datetime import datetime
from sys import getsizeof
from tqdm import tqdm

def generate_embedding_mosaic(mosaic_path, year, region_id, version, output_dir, model, EPOCH=100, patch_size=256, step=128):
    """
    Generates a multi-band embedding/feature mosaic from a large image using a
    smart, flexible, and memory-efficient stitching process with overlap-averaging.
    """
    # --- Sections 1-4 are correct and remain the same ---
    output_dir_with_epoch = os.path.join(output_dir, str(year), str(EPOCH))
    os.makedirs(output_dir_with_epoch, exist_ok=True)
    base_filename = os.path.basename(mosaic_path).replace('.tif', '')
    final_raster_uri = os.path.join(output_dir_with_epoch, f'outimage_v{version}_e{EPOCH}_grid_{region_id}_{year}_embedding_normalized.tif')
    if os.path.exists(final_raster_uri):
        print(f"Final file already exists, skipping: {final_raster_uri}")
        return
    print(f"--- Starting embedding generation for model: '{model.name}' ---")
    with rasterio.open(mosaic_path, 'r') as ds:
        out_meta = ds.meta.copy()
        original_transform = ds.transform
        img_arr = ds.read().astype(np.float32)
    img_arr = np.nan_to_num(np.clip(img_arr, 0, None), nan=0.0)
    if img_arr.shape[0] > 6:
        img_arr = img_arr[1:, :, :]
    img_arr_hwc = np.transpose(img_arr, [1, 2, 0])
    h_original, w_original, _ = img_arr_hwc.shape
    print(f"Original image shape (H, W, C): {(h_original, w_original, 6)}")
    #NP.pad is missing
    patches = patchify(img_arr_hwc, (patch_size, patch_size, 6), step=step) #


    patches_reshaped = patches.reshape(-1, patch_size, patch_size, 6)
    print(f"Image patched into {patches_reshaped.shape[0]} patches.")
    print("Running memory-efficient prediction...")
    patch_dataset = tf.data.Dataset.from_tensor_slices(patches_reshaped).batch(64)
    prediction_list = [model.predict_on_batch(batch) for batch in tqdm(patch_dataset, desc="Predicting Batches")]
    predictions = np.concatenate(prediction_list, axis=0)
    print("Prediction complete. Predictions shape:", predictions.shape)

    # --- Section 5 is correct and remains the same ---
    print("Stitching results with smart scaling and overlap-averaging...")
    grid_rows, grid_cols = patches.shape[0], patches.shape[1]
    _, pred_h, pred_w, pred_c = predictions.shape
    scale_h, scale_w = patch_size / pred_h, patch_size / pred_w
    pred_step_h, pred_step_w = int(step / scale_h), int(step / scale_w)
    assert step % scale_h == 0 and step % scale_w == 0, "Step size must be divisible by scaling factor."
    canvas_h, canvas_w = int(h_original / scale_h), int(w_original / scale_w)
    canvas_shape = (canvas_h, canvas_w, pred_c)
    prediction_canvas = np.zeros(canvas_shape, dtype=np.float32)
    overlap_counter = np.zeros(canvas_shape, dtype=np.int32)
    for k, predicted_patch in enumerate(tqdm(predictions, desc="Stitching Patches")):
        # predicted_patch[0, :, :] = 1;
        # predicted_patch[-1, :, :] = 1
        # predicted_patch[:, 0, :] = 1;
        # predicted_patch[:, -1, :] = 1
        i = k // grid_cols
        j = k % grid_cols
        x_start, y_start = i * pred_step_h, j * pred_step_w
        prediction_canvas[x_start:x_start + pred_h, y_start:y_start + pred_w] += predicted_patch
        overlap_counter[x_start:x_start + pred_h, y_start:y_start + pred_w] += 1
    print("Averaging overlapped regions...")
    overlap_counter[overlap_counter == 0] = 1
    smooth_mosaic_scaled = prediction_canvas / overlap_counter

    # --- 6. CORRECTED: Resize, Normalize, and Save ---

    print("Normalizing final mosaic...")
    min_vals = np.min(smooth_mosaic_scaled, axis=(0, 1))
    max_vals = np.max(smooth_mosaic_scaled, axis=(0, 1))
    range_vals = max_vals - min_vals
    range_vals[range_vals == 0] = 1.0
    normalized_image = (smooth_mosaic_scaled - min_vals) / range_vals
    final_image_uint8 = (normalized_image * 255).astype(np.uint8)

    reconstructed_image_chw = np.transpose(final_image_uint8, [2, 0, 1])
    print("Stitching complete. Final array shape (C, H, W):", reconstructed_image_chw.shape)


    original_gdal_transform = out_meta['transform']
    new_transform = original_gdal_transform * Affine.scale(scale_w, scale_h)

    # --- 7. Save the Final GeoTIFF (This part is correct) ---
    out_meta.update({
    "driver": "GTiff",
    "height": smooth_mosaic_scaled.shape[0],  # Use the SMALL height
    "width": smooth_mosaic_scaled.shape[1],   # Use the SMALL width
    "count": reconstructed_image_chw.shape[0],   # The number of feature channels # BUG encontrado
    "dtype": "uint8",      # Use the float32 dtype of the mosaic
    "transform": new_transform,               # Use our NEW scaled transform
    "compress": 'lzw'
})


    print(f"Saving final output to: {final_raster_uri}")
    with rasterio.open(final_raster_uri, 'w', **out_meta, tiled=True, blockxsize=256, blockysize=256, predictor=2) as dest:
        dest.write(reconstructed_image_chw)

    print("Processing complete!")

In [ ]:
# !pip install pygeoj
import pygeoj
kernel_buffer = [256, 256]
image_base_name = 'allPatch_UNET_grid_'
user_folder = 'projects/samm/Mapbiomas8/'
# Base file name to use for TFRecord files and assets.
grid = pygeoj.load('/mnt/nfs/assets/Mapbiomas/modelos/mb10-unet-mineracao/GRID_ALLCLASSES_COLECAO_10_STACK_BIOMA_V1.geojson')
print("Total Features on GRID")
print(len(grid))
original = [int(geo.properties['id']) for geo in grid]
original.sort()
# print(original)

In [ ]:
def check_file_exists(paths):
    """Verifica se algum dos caminhos especificados já existe."""
    for path in paths:
        if os.path.exists(path):
            return True
    return False
def create_directory(new_folder):
  if not os.path.exists(new_folder):
      print(f'lets make the directory: {new_folder}')
      os.makedirs(new_folder)
  else: return

def dynamic_slice_or_pad(target_shape, predicted_img):
    print(f'ORIGINAL SHAPE: {target_shape}')
    print(f'predicted_img SHAPE: {predicted_img.shape}')
    target_rows, target_cols = target_shape
    pad_rows = target_rows - predicted_img.shape[0]
    pad_cols = target_cols - predicted_img.shape[1]

    # ORIGINAL ROWS IS SMALLER
    if pad_rows < 0:
        predicted_img = predicted_img[:target_rows, :]
    elif pad_rows > 0:
        predicted_img = np.pad(predicted_img, ((0, pad_rows), (0, 0)), mode='constant')

    # ORIGINAL COLUMNS IS SMALLER
    if pad_cols < 0:
        predicted_img = predicted_img[:, :target_cols]
    elif pad_cols > 0:
        predicted_img = np.pad(predicted_img, ((0, 0), (0, pad_cols)), mode='constant')


    print(f'predicted_img SHAPE FINALL  : {predicted_img.shape}')
    return predicted_img

In [ ]:
import time
import glob
import gc
from tqdm import tqdm

gc.collect()
ROOT_PATH = '/mnt/nfs/assets/images'
YEAR      = 2024
MOSAIC_VERSION   = '1'
mosaic_scale = 30

GRIDS_IDS = pygeoj.load('/mnt/nfs/assets/Mapbiomas/modelos/mb10-unet-mineracao/GRID_ALLCLASSES_COLECAO_10_STACK_BIOMA_V1.geojson')
# GRID      = pygeoj.load(f'{ROOT_PATH}/GRIDS/GRID-ALLCALSSES-COL9-STACK.geojson')
loaded_model = new_feature_extractor #model (8x8x571)
reduced_grid = [int(feature.properties['id']) for feature in GRIDS_IDS if feature.properties['mining'] == 1 and feature.properties['CD_Bioma'] == BIOMA_CODE ]
reduced_grid = [int(n + 1) for n in reduced_grid]
reduced_grid.sort()
BIOMA = "pampa"
%pdb off
start = time.time()

for year in range(2023,2024):
    print(f'/n/nYEAR:{year}')
    i = 0
    # add_list=[]
    for region_id in reduced_grid:
        print(f'REGION ID {region_id}')
        tf.keras.backend.clear_session()
        start = time.time()
        # try:
        MOSAIC_PATH  = f'{ROOT_PATH}/mosaico_landsat/{year}'
        search_pattern = os.path.join(MOSAIC_PATH, f'v{MOSAIC_VERSION}_*_grid_{region_id}_{year}*.tif')
        matching_files = glob.glob(search_pattern)

        if matching_files:
            i = i+1
            region_mosaic_file = matching_files[0]
            print("num_classes",num_classes)
            generate_embedding_mosaic(region_mosaic_file,year,region_id,VERSION, OUTPUT_PATH, loaded_model,100, 256, 32)
            #mosaic_predict_multi(region_mosaic_file, year, region_id, OUTPUT_PATH, VERSION, 256, opticalBands, [], loaded_model, 100,num_classes)
        else:
            print(i)
            # logging.error("No matching:",search_pattern)
            # add_list.append(region_id)
            print("No matching:",search_pattern)
        # except Exception as e:
            # add_list.append(region_id)
            # logging.error("The error from mosaic_predict is: ",e)
end = time.time()
# print(add_list)
print('Prediction Time per year = '+str(end - start))

In [ ]:
!gsutil -m cp -r output/v6/* gs://mb10-mining/classifications/mbX/v6/